In [ ]:
%pip install -q polars pyarrow


# Polars Lab: строим хранилище данных с нуля

Этот ноутбук сделан как обучающий практикум для человека, который раньше не работал с хранилищами данных.

Что мы сделаем:
- разберёмся, что такое источник данных и хранилище;
- поймём, что такое `EAV`, `SCD1` и `SCD2`;
- посмотрим на целевую схему таблиц;
- шаг за шагом соберём хранилище из сырых parquet-файлов;
- сохраним результат как набор parquet-таблиц.


## 1. Простая идея: что вообще происходит

У нас есть сырые данные из нескольких источников:
- ежедневный источник по клиентам;
- месячный источник по кредитным картам;
- месячный источник по дебетовым картам.

Эти источники неудобно анализировать напрямую, потому что:
- они лежат в разных папках;
- в них разные наборы колонок;
- часть данных обновляется ежедневно, часть ежемесячно.

Поэтому мы строим **хранилище данных**: приводим всё к одной понятной структуре.

Итогом будут 5 таблиц:
- `dim_sources` — какие есть источники;
- `dim_attributes` — какие есть атрибуты и откуда они пришли;
- `load_log` — что и когда мы загрузили;
- `client_monthly_attrs_scd1` — ежемесячные атрибуты клиентов;
- `client_daily_attrs_scd2` — ежедневные атрибуты клиентов с историей изменений.


## 2. Что такое EAV

`EAV` — это `Entity-Attribute-Value`.

Простыми словами:
- `Entity` — сущность, например клиент;
- `Attribute` — признак, например доход;
- `Value` — значение признака, например `150000`.

Вместо широкой таблицы:

| client_id | income | city | payroll_flag |
|---|---:|---|---:|
| 1 | 150000 | Moscow | 1 |

мы делаем узкую таблицу:

| client_id | attribute_id | attribute_value |
|---:|---:|---|
| 1 | 1 | 150000 |
| 1 | 2 | Moscow |
| 1 | 3 | 1 |

Зачем это удобно:
- можно хранить очень много атрибутов в одной таблице;
- легко объединять разные источники;
- все значения хранятся единообразно.


## 3. Что такое SCD1 и SCD2

Это два способа хранить изменения данных во времени.

### SCD1
Если значение изменилось, мы просто перезаписываем старое значение новым.

Пример:
- было: доход = `100000`
- стало: доход = `120000`
- в таблице останется только `120000`

### SCD2
Если значение изменилось, мы сохраняем историю.

Пример:
- с `2023-01-01` по `2023-03-01`: статус = `A`
- с `2023-03-02` по `9999-12-31`: статус = `B`

В нашем ноутбуке:
- monthly-таблица хранится как `SCD1`;
- daily-таблица хранится как `SCD2`.


## 4. Целевая схема хранилища

Мы будем собирать данные ровно в такую физическую модель:

- `dim_sources`
  - `source_id INT`
  - `source_name STRING`
  - `source_description STRING`
  - `update_frequency STRING`
  - `row_create_dtime DATETIME`
  - `row_update_dtime DATETIME`
  - `valid_from DATETIME`
  - `valid_to DATETIME`

- `dim_attributes`
  - `attribute_id INT`
  - `attribute_name STRING`
  - `attribute_description STRING`
  - `data_type STRING`
  - `source_id INT`
  - `update_frequency STRING`
  - `row_create_dtime DATETIME`
  - `row_update_dtime DATETIME`

- `load_log`
  - `load_id BIGINT`
  - `source_id INT`
  - `source_report_dt STRING`
  - `load_start_dtime DATETIME`
  - `load_end_dtime DATETIME`
  - `target_table STRING`
  - `load_status STRING`
  - `rows_loaded BIGINT`
  - `error_message STRING`

- `client_monthly_attrs_scd1`
  - `client_id INT`
  - `attribute_id INT`
  - `report_dt STRING`
  - `attribute_value STRING`
  - `source_id INT`
  - `row_update_dtime DATETIME`
  - `row_loading_id BIGINT`
  - `row_hash_val STRING`

- `client_daily_attrs_scd2`
  - `client_id INT`
  - `attribute_id INT`
  - `attribute_value STRING`
  - `row_actual_from STRING`
  - `row_actual_to STRING`
  - `source_id INT`
  - `row_update_dtime DATETIME`
  - `row_loading_id BIGINT`
  - `row_hash_val STRING`


## 5. Импорты, пути и подготовка данных

Ниже мы:
- импортируем библиотеки;
- при необходимости распакуем `data.zip` в Colab;
- зададим пути к исходным и выходным данным;
- опишем источники и их бизнес-колонки.


In [42]:
# %pip install polars pyarrow

import datetime as dt
from pathlib import Path
import zipfile

import polars as pl

# Если в Colab загружен data.zip, распакуем его в /content/dbscoring.
for zip_path in (Path("/content/data.zip"), Path("data.zip")):
    if zip_path.exists():
        extract_root = Path("/content/dbscoring")
        extract_root.mkdir(parents=True, exist_ok=True)
        if zipfile.is_zipfile(zip_path):
            with zipfile.ZipFile(zip_path) as archive:
                archive.extractall(extract_root)
            print(f"Extracted {zip_path} to {extract_root}")
        else:
            print(f"Skip unpack: {zip_path} is not a valid zip archive")
        break

# Путь к сырым данным: сначала локальные варианты, потом Colab.
for candidate in (Path("../data/sources"), Path("data/sources"), Path("/content/dbscoring/data/sources")):
    if candidate.exists():
        BASE_DIR = candidate
        break
else:
    BASE_DIR = Path("data/sources")

# Путь к выходным parquet-таблицам: локально пишем локально, в Colab — в /content.
for candidate in (Path("../data/warehouse_polars"), Path("data/warehouse_polars"), Path("/content/dbscoring/data/warehouse_polars")):
    if candidate.exists() or candidate.parent.exists():
        WAREHOUSE_ROOT = candidate
        break
else:
    WAREHOUSE_ROOT = Path("data/warehouse_polars")

NOW_TS = dt.datetime.now().replace(microsecond=0)
START_TS = dt.datetime(2023, 1, 1, 0, 0, 0)
FUTURE_TS = dt.datetime(9999, 12, 31, 0, 0, 0)

# Реестр источников: отсюда мы знаем, какие колонки к какому источнику относятся.
SOURCE_REGISTRY = {
    "client_cards_daily": {
        "source_id": 1,
        "source_description": "Daily client card source.",
        "update_frequency": "daily",
        "target_table": "client_daily_attrs_scd2",
        "columns": (
            "srv_mb_nflag",
            "cc_stoplist",
            "lne_tot_debt_int_ovrd_rub_amt",
            "lne_tot_debt_ovrd_rub_amt",
        ),
    },
    "credit_cards_info": {
        "source_id": 2,
        "source_description": "Monthly credit card source.",
        "update_frequency": "monthly",
        "target_table": "client_monthly_attrs_scd1",
        "columns": (
            "client_income_amt",
            "oi_total_amt",
            "act_pl_os_rub_amt",
            "payroll_client_nflag",
            "inf_payroll_rub_amt",
            "legal_entity_amt",
            "inc_avg_risk_rub_amt",
            "otf_loan_rub_amt",
            "otf_fee_rub_amt",
            "inf_transfer_rub_amt",
            "cc_ever_nflag",
        ),
    },
    "deb_cards_info": {
        "source_id": 3,
        "source_description": "Monthly debit card source.",
        "update_frequency": "monthly",
        "target_table": "client_monthly_attrs_scd1",
        "columns": (
            "onl_bank_active_1m_nfalg",
            "auto_pay_active_qty",
            "cl_income_1m_amt",
            "dep_acc_1st_open_dt",
            "wdr_cash_6m_amt",
            "cash_op_6m_amt",
            "cash_3m_qty",
            "lst_balance_amt",
            "card_active_1m_nflag",
        ),
    },
}

# Всем атрибутам заранее назначаем целочисленные id.
ALL_COLUMNS = [column_name for meta in SOURCE_REGISTRY.values() for column_name in meta["columns"]]
ATTR_ID_BY_NAME = {name: idx for idx, name in enumerate(ALL_COLUMNS, start=1)}

print(BASE_DIR)
print(WAREHOUSE_ROOT)


../data/sources
../data/warehouse_polars


## 6. Что такое схема таблицы в Polars

Схема таблицы — это описание её колонок и типов.

Например:
- `client_id: pl.Int32` означает, что `client_id` хранится как целое число;
- `attribute_value: pl.String` означает, что значение атрибута хранится как строка.

Ниже мы задаём схемы сразу для всех 5 таблиц хранилища.


In [43]:
DIM_SOURCES_SCHEMA = {
    "source_id": pl.Int32,
    "source_name": pl.String,
    "source_description": pl.String,
    "update_frequency": pl.String,
    "row_create_dtime": pl.Datetime,
    "row_update_dtime": pl.Datetime,
    "valid_from": pl.Datetime,
    "valid_to": pl.Datetime,
}

DIM_ATTRIBUTES_SCHEMA = {
    "attribute_id": pl.Int32,
    "attribute_name": pl.String,
    "attribute_description": pl.String,
    "data_type": pl.String,
    "source_id": pl.Int32,
    "update_frequency": pl.String,
    "row_create_dtime": pl.Datetime,
    "row_update_dtime": pl.Datetime,
}

LOAD_LOG_SCHEMA = {
    "load_id": pl.Int64,
    "source_id": pl.Int32,
    "source_report_dt": pl.String,
    "load_start_dtime": pl.Datetime,
    "load_end_dtime": pl.Datetime,
    "target_table": pl.String,
    "load_status": pl.String,
    "rows_loaded": pl.Int64,
    "error_message": pl.String,
}

MONTHLY_SCHEMA = {
    "client_id": pl.Int32,
    "attribute_id": pl.Int32,
    "report_dt": pl.String,
    "attribute_value": pl.String,
    "source_id": pl.Int32,
    "row_update_dtime": pl.Datetime,
    "row_loading_id": pl.Int64,
    "row_hash_val": pl.String,
}

DAILY_SCHEMA = {
    "client_id": pl.Int32,
    "attribute_id": pl.Int32,
    "attribute_value": pl.String,
    "row_actual_from": pl.String,
    "row_actual_to": pl.String,
    "source_id": pl.Int32,
    "row_update_dtime": pl.Datetime,
    "row_loading_id": pl.Int64,
    "row_hash_val": pl.String,
}


## 7. Читаем реальные данные

Мы будем работать не со всеми возможными данными, а с конкретными срезами:
- daily: один старый и один актуальный срез;
- monthly: февраль и март 2023 года.

Это удобно для учебной демонстрации: видно, как строятся итоговые таблицы, но логика остаётся реалистичной.


In [44]:
def read_partition(path: Path) -> pl.DataFrame:
    """Читает все parquet-файлы из одной партиции."""
    files = sorted(path.glob("*.parquet"))
    if not files:
        raise FileNotFoundError(path)
    return pl.read_parquet(files)

ccd_2023 = read_partition(BASE_DIR / "client_cards_daily" / "row_actual_to='2023-04-03'")
ccd_9999 = read_partition(BASE_DIR / "client_cards_daily" / "row_actual_to='9999-12-31'")
cci_02 = read_partition(BASE_DIR / "credit_cards_info" / "report_dt='2023-02-28'")
cci_03 = read_partition(BASE_DIR / "credit_cards_info" / "report_dt='2023-03-31'")
dci_02 = read_partition(BASE_DIR / "deb_cards_info" / "report_dt='2023-02-28'")
dci_03 = read_partition(BASE_DIR / "deb_cards_info" / "report_dt='2023-03-31'")

print("ccd_2023:", ccd_2023.shape)
print("ccd_9999:", ccd_9999.shape)
print("cci_02:", cci_02.shape)
print("cci_03:", cci_03.shape)
print("dci_02:", dci_02.shape)
print("dci_03:", dci_03.shape)


ccd_2023: (751, 10)
ccd_9999: (189690, 10)
cci_02: (286339, 16)
cci_03: (286339, 16)
dci_02: (285945, 14)
dci_03: (285420, 14)


## 8. Посмотрим на wide-формат источника

Сейчас данные ещё в широком виде: одна строка содержит много разных колонок клиента.

Это удобно для операционной системы источника, но неудобно для нашей универсальной модели EAV.

Ниже можно посмотреть несколько колонок monthly-источника.


In [45]:
cci_02.select([
    "id",
    "report_dt",
    "client_income_amt",
    "oi_total_amt",
    "cc_ever_nflag",
]).head(5)


id,report_dt,client_income_amt,oi_total_amt,cc_ever_nflag
str,str,"decimal[18,2]","decimal[20,2]",i32
"""0021fe6fa468322c49b840c13b457e…","""2023-02-28""",null,0.45,1
"""00308054c2e81501d3121e153dfff9…","""2023-02-28""",null,0.25,0
"""000ce43f10410f3fc4d52ac464ade2…","""2023-02-28""",null,2.24,0
"""00034be866d9bcd92c4f5cfab1f69d…","""2023-02-28""",null,50.92,1
"""0031929ac0d5ebaac27fb56cab2107…","""2023-02-28""",null,853.50,0


## 9. Маленький игрушечный пример EAV на 2 строках

Прежде чем переходить к миллионам строк, покажем идею на маленьком примере.


In [46]:
toy = pl.DataFrame({
    "id": ["client_a", "client_b"],
    "income": [100000, 120000],
    "city": ["Moscow", "Kazan"],
})

toy


id,income,city
str,i64,str
"""client_a""",100000,"""Moscow"""
"""client_b""",120000,"""Kazan"""


In [47]:
toy.unpivot(
    index=["id"],
    on=["income", "city"],
    variable_name="attribute_name",
    value_name="attribute_value",
)


id,attribute_name,attribute_value
str,str,str
"""client_a""","""income""","""100000"""
"""client_b""","""income""","""120000"""
"""client_a""","""city""","""Moscow"""
"""client_b""","""city""","""Kazan"""


Видно, что одна широкая строка превращается в несколько строк вида:
- какой клиент;
- какой атрибут;
- какое значение.

Теперь сделаем то же самое на реальных данных.


## 10. Вспомогательные функции

Ниже — функции, которые будут строить итоговые таблицы.

Не пугайся: после определения мы пройдёмся по их результатам шаг за шагом.


In [48]:
# Собираем отображение raw string id -> surrogate int client_id.
CLIENT_ID_MAP = {
    raw_id: idx
    for idx, raw_id in enumerate(
        sorted(
            set(pl.concat([
                ccd_2023.select(pl.col("id").cast(pl.String)),
                ccd_9999.select(pl.col("id").cast(pl.String)),
                cci_02.select(pl.col("id").cast(pl.String)),
                cci_03.select(pl.col("id").cast(pl.String)),
                dci_02.select(pl.col("id").cast(pl.String)),
                dci_03.select(pl.col("id").cast(pl.String)),
            ], how="vertical")["id"].to_list())
        ),
        start=1,
    )
}

def infer_data_type(name: str) -> str:
    """Грубо определяет тип атрибута по имени колонки."""
    if name.endswith("_dt") or name.endswith("_from") or name.endswith("_to"):
        return "DATE"
    if name.endswith("_amt"):
        return "DECIMAL"
    if name.endswith("_nflag") or name.endswith("_nfalg") or name.endswith("_qty"):
        return "INT"
    return "STRING"

def build_dim_sources() -> pl.DataFrame:
    """Собирает справочник источников dim_sources."""
    rows = [{
        "source_id": meta["source_id"],
        "source_name": source_name,
        "source_description": meta["source_description"],
        "update_frequency": meta["update_frequency"],
        "row_create_dtime": NOW_TS,
        "row_update_dtime": NOW_TS,
        "valid_from": START_TS,
        "valid_to": FUTURE_TS,
    } for source_name, meta in SOURCE_REGISTRY.items()]
    return pl.DataFrame(rows, schema=DIM_SOURCES_SCHEMA)

def build_dim_attributes() -> pl.DataFrame:
    """Собирает справочник атрибутов dim_attributes."""
    rows = []
    for source_name, meta in SOURCE_REGISTRY.items():
        for column_name in meta["columns"]:
            rows.append({
                "attribute_id": ATTR_ID_BY_NAME[column_name],
                "attribute_name": column_name,
                "attribute_description": f"Attribute {column_name}",
                "data_type": infer_data_type(column_name),
                "source_id": meta["source_id"],
                "update_frequency": meta["update_frequency"],
                "row_create_dtime": NOW_TS,
                "row_update_dtime": NOW_TS,
            })
    return pl.DataFrame(rows, schema=DIM_ATTRIBUTES_SCHEMA)

def unpivot_table(df: pl.DataFrame, source_name: str) -> pl.DataFrame:
    """Разворачивает wide-таблицу источника в EAV и приводит поля к целевой схеме."""
    meta = SOURCE_REGISTRY[source_name]

    # Все business-значения заранее приводим к строке.
    prepared = df.with_columns([
        pl.col(column_name).cast(pl.String) for column_name in meta["columns"]
    ])

    # Из wide-формата получаем пары attribute_name / attribute_value.
    melted = prepared.unpivot(
        index=["id", "row_update_dtime", "loading_id", "row_hash_val"] + (["report_dt"] if meta["update_frequency"] == "monthly" else ["row_actual_from", "row_actual_to"]),
        on=list(meta["columns"]),
        variable_name="attribute_name",
        value_name="attribute_value",
    ).with_columns(
        pl.col("id").cast(pl.String).replace_strict(CLIENT_ID_MAP).cast(pl.Int32).alias("client_id"),
        pl.col("attribute_name").replace_strict(ATTR_ID_BY_NAME).cast(pl.Int32).alias("attribute_id"),
        pl.col("attribute_value").cast(pl.String),
        pl.col("row_update_dtime").cast(pl.Datetime),
        pl.col("loading_id").cast(pl.Int64).alias("row_loading_id"),
        pl.lit(meta["source_id"]).cast(pl.Int32).alias("source_id"),
        pl.col("row_hash_val").cast(pl.String),
    )

    if meta["update_frequency"] == "monthly":
        return melted.select([
            pl.col("client_id"),
            pl.col("attribute_id"),
            pl.col("report_dt").cast(pl.String),
            pl.col("attribute_value"),
            pl.col("source_id"),
            pl.col("row_update_dtime"),
            pl.col("row_loading_id"),
            pl.col("row_hash_val"),
        ])

    return melted.select([
        pl.col("client_id"),
        pl.col("attribute_id"),
        pl.col("attribute_value"),
        pl.col("row_actual_from").cast(pl.String),
        pl.col("row_actual_to").cast(pl.String),
        pl.col("source_id"),
        pl.col("row_update_dtime"),
        pl.col("row_loading_id"),
        pl.col("row_hash_val"),
    ])


## 11. Строим `dim_sources`

Это маленькая таблица-справочник: она просто перечисляет источники данных.


In [49]:
dim_sources = build_dim_sources()
dim_sources


source_id,source_name,source_description,update_frequency,row_create_dtime,row_update_dtime,valid_from,valid_to
i32,str,str,str,datetime[μs],datetime[μs],datetime[μs],datetime[μs]
1,"""client_cards_daily""","""Daily client card source.""","""daily""",2026-05-13 20:25:32,2026-05-13 20:25:32,2023-01-01 00:00:00,9999-12-31 00:00:00
2,"""credit_cards_info""","""Monthly credit card source.""","""monthly""",2026-05-13 20:25:32,2026-05-13 20:25:32,2023-01-01 00:00:00,9999-12-31 00:00:00
3,"""deb_cards_info""","""Monthly debit card source.""","""monthly""",2026-05-13 20:25:32,2026-05-13 20:25:32,2023-01-01 00:00:00,9999-12-31 00:00:00


## 12. Строим `dim_attributes`

Здесь каждая строка — это один бизнес-атрибут.

Например:
- у атрибута есть `attribute_id`;
- имя атрибута, например `client_income_amt`;
- тип данных;
- источник, из которого он пришёл.


In [50]:
dim_attributes = build_dim_attributes()
dim_attributes.head(10)


attribute_id,attribute_name,attribute_description,data_type,source_id,update_frequency,row_create_dtime,row_update_dtime
i32,str,str,str,i32,str,datetime[μs],datetime[μs]
1,"""srv_mb_nflag""","""Attribute srv_mb_nflag""","""INT""",1,"""daily""",2026-05-13 20:25:32,2026-05-13 20:25:32
2,"""cc_stoplist""","""Attribute cc_stoplist""","""STRING""",1,"""daily""",2026-05-13 20:25:32,2026-05-13 20:25:32
3,"""lne_tot_debt_int_ovrd_rub_amt""","""Attribute lne_tot_debt_int_ovr…","""DECIMAL""",1,"""daily""",2026-05-13 20:25:32,2026-05-13 20:25:32
4,"""lne_tot_debt_ovrd_rub_amt""","""Attribute lne_tot_debt_ovrd_ru…","""DECIMAL""",1,"""daily""",2026-05-13 20:25:32,2026-05-13 20:25:32
5,"""client_income_amt""","""Attribute client_income_amt""","""DECIMAL""",2,"""monthly""",2026-05-13 20:25:32,2026-05-13 20:25:32
6,"""oi_total_amt""","""Attribute oi_total_amt""","""DECIMAL""",2,"""monthly""",2026-05-13 20:25:32,2026-05-13 20:25:32
7,"""act_pl_os_rub_amt""","""Attribute act_pl_os_rub_amt""","""DECIMAL""",2,"""monthly""",2026-05-13 20:25:32,2026-05-13 20:25:32
8,"""payroll_client_nflag""","""Attribute payroll_client_nflag""","""INT""",2,"""monthly""",2026-05-13 20:25:32,2026-05-13 20:25:32
9,"""inf_payroll_rub_amt""","""Attribute inf_payroll_rub_amt""","""DECIMAL""",2,"""monthly""",2026-05-13 20:25:32,2026-05-13 20:25:32


## 13. Разворачиваем monthly-источник в EAV

Ниже можно увидеть, как wide-таблица превращается в строки вида:
- `client_id`
- `attribute_id`
- `report_dt`
- `attribute_value`


In [51]:
monthly_example = unpivot_table(cci_02.head(3), "credit_cards_info")
monthly_example.head(12)


client_id,attribute_id,report_dt,attribute_value,source_id,row_update_dtime,row_loading_id,row_hash_val
i32,i32,str,str,i32,datetime[μs],i64,str
106020,5,"""2023-02-28""",null,2,2023-02-28 10:45:23.340,345812345,"""01ee5be65a42121beaf62f9fb0047c…"
151256,5,"""2023-02-28""",null,2,2023-02-28 10:45:23.340,345812345,"""a9a64f6df3d2776ce0cf4df59cada5…"
40121,5,"""2023-02-28""",null,2,2023-02-28 10:45:23.340,345812345,"""b1976be321fa17439df6a953b7ae23…"
106020,6,"""2023-02-28""","""0.45""",2,2023-02-28 10:45:23.340,345812345,"""01ee5be65a42121beaf62f9fb0047c…"
151256,6,"""2023-02-28""","""0.25""",2,2023-02-28 10:45:23.340,345812345,"""a9a64f6df3d2776ce0cf4df59cada5…"
…,…,…,…,…,…,…,…
151256,7,"""2023-02-28""",null,2,2023-02-28 10:45:23.340,345812345,"""a9a64f6df3d2776ce0cf4df59cada5…"
40121,7,"""2023-02-28""",null,2,2023-02-28 10:45:23.340,345812345,"""b1976be321fa17439df6a953b7ae23…"
106020,8,"""2023-02-28""","""0""",2,2023-02-28 10:45:23.340,345812345,"""01ee5be65a42121beaf62f9fb0047c…"


## 14. Разворачиваем daily-источник в EAV

Для daily-таблицы сохраняются даты действия записи:
- `row_actual_from`
- `row_actual_to`

Именно поэтому эту таблицу удобно трактовать как `SCD2`.


In [52]:
daily_example = unpivot_table(ccd_2023.head(3), "client_cards_daily")
daily_example.head(12)


client_id,attribute_id,attribute_value,row_actual_from,row_actual_to,source_id,row_update_dtime,row_loading_id,row_hash_val
i32,i32,str,str,str,i32,datetime[μs],i64,str
94314,1,null,"""2023-04-03""","""2023-04-03""",1,2023-04-04 09:12:45.330,23423311,"""cf99efcc3bca1ae0a8948798914301…"
36256,1,null,"""2023-04-03""","""2023-04-03""",1,2023-04-04 09:12:45.330,23423311,"""15192bfbb3c806ddb27c801af52539…"
13325,1,null,"""2023-04-03""","""2023-04-03""",1,2023-04-04 09:12:45.330,23423311,"""fc6283c8b9a95c99ab0ed12a0e921a…"
94314,2,"""1""","""2023-04-03""","""2023-04-03""",1,2023-04-04 09:12:45.330,23423311,"""cf99efcc3bca1ae0a8948798914301…"
36256,2,"""1""","""2023-04-03""","""2023-04-03""",1,2023-04-04 09:12:45.330,23423311,"""15192bfbb3c806ddb27c801af52539…"
…,…,…,…,…,…,…,…,…
36256,3,null,"""2023-04-03""","""2023-04-03""",1,2023-04-04 09:12:45.330,23423311,"""15192bfbb3c806ddb27c801af52539…"
13325,3,null,"""2023-04-03""","""2023-04-03""",1,2023-04-04 09:12:45.330,23423311,"""fc6283c8b9a95c99ab0ed12a0e921a…"
94314,4,null,"""2023-04-03""","""2023-04-03""",1,2023-04-04 09:12:45.330,23423311,"""cf99efcc3bca1ae0a8948798914301…"


## 15. Собираем итоговую monthly SCD1-таблицу

Что здесь происходит:
- берём 4 monthly-среза;
- разворачиваем их в EAV;
- объединяем в одну таблицу;
- убираем дубли по ключу:
  - `client_id`
  - `attribute_id`
  - `report_dt`


In [53]:
client_monthly_attrs_scd1 = pl.concat([
    unpivot_table(cci_02, "credit_cards_info"),
    unpivot_table(cci_03, "credit_cards_info"),
    unpivot_table(dci_02, "deb_cards_info"),
    unpivot_table(dci_03, "deb_cards_info"),
], how="vertical").unique(subset=["client_id", "attribute_id", "report_dt"], keep="last").cast(MONTHLY_SCHEMA)

client_monthly_attrs_scd1.head(10)


client_id,attribute_id,report_dt,attribute_value,source_id,row_update_dtime,row_loading_id,row_hash_val
i32,i32,str,str,i32,datetime[μs],i64,str
192765,8,"""2023-02-28""","""0""",2,2023-02-28 10:45:23.340,345812345,"""e5fb7bbb69c3cf4e8ceb9b5048b47b…"
123462,9,"""2023-02-28""",null,2,2023-02-28 10:45:23.340,345812345,"""020a194f6d4816238d3b21e42efa16…"
160147,7,"""2023-03-31""",null,2,2023-03-31 14:23:13.560,342671978,"""bb14b3a2ab1c4cca354f2e36f07e7d…"
109488,15,"""2023-03-31""","""0""",2,2023-03-31 14:23:13.560,342671978,"""16109a5e5e3b246c9cf3f7acf770aa…"
275938,6,"""2023-02-28""","""0.24""",2,2023-02-28 10:45:23.340,345812345,"""20c19b00aba44d417d2761a6dc4d2d…"
193375,11,"""2023-02-28""",null,2,2023-02-28 10:45:23.340,345812345,"""810119546d35bdf517914f447035fe…"
129401,23,"""2023-03-31""",null,3,2023-03-31 12:35:25.650,45653454,"""a856b410db6fed10cd7bb5404c38b0…"
205447,11,"""2023-02-28""","""30624.00""",2,2023-02-28 10:45:23.340,345812345,"""ac1fcb43c427f5207af057a08b0eed…"
238436,17,"""2023-02-28""",null,3,2023-02-28 10:34:23.780,45633434,"""755d420532a3fd0074f0776437b92d…"


## 16. Собираем итоговую daily SCD2-таблицу

Что здесь происходит:
- берём 2 daily-среза;
- разворачиваем их в EAV;
- объединяем в одну таблицу;
- убираем дубли по ключу:
  - `client_id`
  - `attribute_id`
  - `row_actual_from`

История значений хранится за счёт полей `row_actual_from` и `row_actual_to`.


In [54]:
client_daily_attrs_scd2 = pl.concat([
    unpivot_table(ccd_2023, "client_cards_daily"),
    unpivot_table(ccd_9999, "client_cards_daily"),
], how="vertical").unique(subset=["client_id", "attribute_id", "row_actual_from"], keep="last").cast(DAILY_SCHEMA)

client_daily_attrs_scd2.head(10)


client_id,attribute_id,attribute_value,row_actual_from,row_actual_to,source_id,row_update_dtime,row_loading_id,row_hash_val
i32,i32,str,str,str,i32,datetime[μs],i64,str
256880,2,"""1""","""2023-04-04""","""9999-12-31""",1,2023-04-04 09:12:45.330,23423311,"""f628e1202d9d64ed2368aa44975d84…"
13102,3,null,"""2023-04-04""","""9999-12-31""",1,2023-04-04 09:12:45.330,23423311,"""7cac11c9885b85915b30560617a6be…"
243976,4,null,"""2023-04-04""","""9999-12-31""",1,2023-04-04 09:12:45.330,23423311,"""e1e339e8a20758bd1a6115a7e4d57b…"
76320,4,null,"""2023-04-04""","""9999-12-31""",1,2023-04-04 09:12:45.330,23423311,"""a900dfbafec84f40b1d8f094784726…"
133163,4,"""0.00""","""2023-04-04""","""9999-12-31""",1,2023-04-04 09:12:45.330,23423311,"""4b78a9030666ccbd3dc28860aa1c00…"
171658,4,"""0.00""","""2023-04-04""","""9999-12-31""",1,2023-04-04 09:12:45.330,23423311,"""cabd570974aede42accf23f051435a…"
162657,2,"""1""","""2023-04-04""","""9999-12-31""",1,2023-04-04 09:12:45.330,23423311,"""182bb3a0f7f72330cef5e444443cea…"
180022,3,null,"""2023-04-04""","""9999-12-31""",1,2023-04-04 09:12:45.330,23423311,"""fb5ba1a730b51f13f9b8801829f94b…"
52089,3,null,"""2023-04-04""","""9999-12-31""",1,2023-04-04 09:12:45.330,23423311,"""2b0840e5f7ed8adf3f5c7c447481ae…"


## 17. Формируем `load_log`

Эта таблица отвечает на вопрос:
- что именно мы загрузили;
- из какого источника;
- на какую дату среза;
- сколько строк прочитали.


In [55]:
load_log = pl.DataFrame([
    {"load_id": 1, "source_id": 1, "source_report_dt": "2023-04-03", "load_start_dtime": NOW_TS, "load_end_dtime": NOW_TS, "target_table": "client_daily_attrs_scd2", "load_status": "success", "rows_loaded": ccd_2023.height, "error_message": None},
    {"load_id": 2, "source_id": 1, "source_report_dt": "9999-12-31", "load_start_dtime": NOW_TS, "load_end_dtime": NOW_TS, "target_table": "client_daily_attrs_scd2", "load_status": "success", "rows_loaded": ccd_9999.height, "error_message": None},
    {"load_id": 3, "source_id": 2, "source_report_dt": "2023-02-28", "load_start_dtime": NOW_TS, "load_end_dtime": NOW_TS, "target_table": "client_monthly_attrs_scd1", "load_status": "success", "rows_loaded": cci_02.height, "error_message": None},
    {"load_id": 4, "source_id": 2, "source_report_dt": "2023-03-31", "load_start_dtime": NOW_TS, "load_end_dtime": NOW_TS, "target_table": "client_monthly_attrs_scd1", "load_status": "success", "rows_loaded": cci_03.height, "error_message": None},
    {"load_id": 5, "source_id": 3, "source_report_dt": "2023-02-28", "load_start_dtime": NOW_TS, "load_end_dtime": NOW_TS, "target_table": "client_monthly_attrs_scd1", "load_status": "success", "rows_loaded": dci_02.height, "error_message": None},
    {"load_id": 6, "source_id": 3, "source_report_dt": "2023-03-31", "load_start_dtime": NOW_TS, "load_end_dtime": NOW_TS, "target_table": "client_monthly_attrs_scd1", "load_status": "success", "rows_loaded": dci_03.height, "error_message": None},
], schema=LOAD_LOG_SCHEMA)

load_log


load_id,source_id,source_report_dt,load_start_dtime,load_end_dtime,target_table,load_status,rows_loaded,error_message
i64,i32,str,datetime[μs],datetime[μs],str,str,i64,str
1,1,"""2023-04-03""",2026-05-13 20:25:32,2026-05-13 20:25:32,"""client_daily_attrs_scd2""","""success""",751,null
2,1,"""9999-12-31""",2026-05-13 20:25:32,2026-05-13 20:25:32,"""client_daily_attrs_scd2""","""success""",189690,null
3,2,"""2023-02-28""",2026-05-13 20:25:32,2026-05-13 20:25:32,"""client_monthly_attrs_scd1""","""success""",286339,null
4,2,"""2023-03-31""",2026-05-13 20:25:32,2026-05-13 20:25:32,"""client_monthly_attrs_scd1""","""success""",286339,null
5,3,"""2023-02-28""",2026-05-13 20:25:32,2026-05-13 20:25:32,"""client_monthly_attrs_scd1""","""success""",285945,null
6,3,"""2023-03-31""",2026-05-13 20:25:32,2026-05-13 20:25:32,"""client_monthly_attrs_scd1""","""success""",285420,null


## 18. Сохраняем хранилище в parquet

Теперь все 5 таблиц готовы. Осталось сохранить их на диск.


In [56]:
WAREHOUSE_ROOT.mkdir(parents=True, exist_ok=True)

dim_sources.write_parquet(WAREHOUSE_ROOT / "dim_sources.parquet")
dim_attributes.write_parquet(WAREHOUSE_ROOT / "dim_attributes.parquet")
load_log.write_parquet(WAREHOUSE_ROOT / "load_log.parquet")
client_monthly_attrs_scd1.write_parquet(WAREHOUSE_ROOT / "client_monthly_attrs_scd1.parquet")
client_daily_attrs_scd2.write_parquet(WAREHOUSE_ROOT / "client_daily_attrs_scd2.parquet")

print(WAREHOUSE_ROOT)


../data/warehouse_polars


## 19. Итог: что получилось

После запуска ноутбука произошло вот что:
- мы прочитали сырые parquet-срезы;
- построили справочник источников;
- построили справочник атрибутов;
- развернули wide-данные в EAV-формат;
- собрали monthly SCD1-таблицу;
- собрали daily SCD2-таблицу;
- создали журнал загрузки;
- сохранили все 5 таблиц как parquet.

То есть по сути мы построили маленькое учебное хранилище данных поверх сырых источников.


In [57]:
{
    "dim_sources": dim_sources.height,
    "dim_attributes": dim_attributes.height,
    "load_log": load_log.height,
    "client_monthly_attrs_scd1": client_monthly_attrs_scd1.height,
    "client_daily_attrs_scd2": client_daily_attrs_scd2.height,
}


{'dim_sources': 3,
 'dim_attributes': 24,
 'load_log': 6,
 'client_monthly_attrs_scd1': 11441743,
 'client_daily_attrs_scd2': 761764}